In [4]:
#!pip install openbabel
!pip install py3Dmol

In [5]:
#import openbabel
import py3Dmol

Objetivo: Mostrar la superficie electroestática de una proteína para analizar regiones cargadas.
1.  Carga una proteína (por ejemplo, 1AKI, una kinasa).
2.  Usa py3Dmol para mostrar la estructura en cartoon y añade una superficie electroestática.
3.  Resalta el sitio activo con un estilo diferente.

In [31]:
import py3Dmol

# Crear visor y cargar proteína
viewer = py3Dmol.view(query='pdb:1A28', width=600, height=400)

# Estilo cartoon para la estructura secundaria
viewer.setStyle({'cartoon': {'color': 'white'}})

# Añadir superficie electroestática
viewer.addSurface(py3Dmol.SES, {'opacity': 0.8, 'colorscheme': {'prop': 'partialCharge', 'gradient': 'roygb'}}) #'gradient': 'rwb'

# Resaltar sitio activo (residuos seleccionados, por ejemplo, 139-142)
viewer.addStyle({'resi': '139-142', 'chain': 'A'}, {'stick': {'colorscheme': 'yellowCarbon'}})

# Mostrar
viewer.zoomTo()
viewer.setBackgroundColor('black')
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Objetivo: Identificar y resaltar interacciones específicas entre una proteína y un ligando.
1.  Carga una proteína con un ligando (por ejemplo, 3ERT, receptor de estrógeno con un ligando).
2.  Resalta los residuos involucrados en puentes de hidrógeno y contactos hidrófobos.
3.  Usa líneas para mostrar las interacciones.

In [7]:
import py3Dmol

# Crear visor y cargar proteína
viewer = py3Dmol.view(query='pdb:3RO5', width=600, height=400)

# Estilo para la proteína
viewer.setStyle({'cartoon': {'color': 'lightblue'}})



# Resaltar residuos cercanos (contactos hidrófobos, < 4 Å)
viewer.addStyle({'within': {'distance': 4.0, 'sel': {'hetflag': True}}},
                {'stick': {'colorscheme': 'greenCarbon'}})



# Estilo para el ligando
viewer.addStyle({'hetflag': True}, {'stick': {'colorscheme': 'redCarbon'}})


# Mostrar puentes de hidrógeno como líneas
viewer.addStyle({'within': {'distance': 3.5, 'sel': {'hetflag': True}}},
                {'line': {'colorscheme': 'black'}})
# Mostrar
viewer.zoomTo()
viewer.setBackgroundColor('white')
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Objetivo: Superponer dos proteínas (2IQH y 3RO5) alineándolas con Biopython y visualizarlas en py3Dmol con colores diferentes para comparar sus estructuras.

In [8]:
!pip install biopython

import py3Dmol
import requests
from Bio import PDB
from Bio.PDB import Superimposer, PDBIO
import io

# Función para descargar datos PDB
def get_pdb_data(pdb_id):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"No se pudo descargar {pdb_id}")
    return response.text

# Descargar datos PDB
pdb_data_1ubq = get_pdb_data('1UBQ')
pdb_data_2y6w = get_pdb_data('2Y6W')

# Guardar datos en archivos temporales para Biopython
with open('1ubq.pdb', 'w') as f:
    f.write(pdb_data_1ubq)
with open('2y6w.pdb', 'w') as f:
    f.write(pdb_data_2y6w)

# Parsear estructuras con Biopython
parser = PDB.PDBParser(QUIET=True)
structure_1ubq = parser.get_structure('1UBQ', '1ubq.pdb')
structure_2y6w = parser.get_structure('2Y6W', '2y6w.pdb')

# Obtener átomos de la cadena A (puedes ajustar según la estructura)
atoms_1ubq = [atom for atom in structure_1ubq[0]['A'].get_atoms() if atom.get_name() == 'CA']  # Solo átomos CA
atoms_2y6w = [atom for atom in structure_2y6w[0]['A'].get_atoms() if atom.get_name() == 'CA']

# Asegurar que ambas listas tengan la misma longitud
min_len = min(len(atoms_1ubq), len(atoms_2y6w))
atoms_1ubq = atoms_1ubq[:min_len]
atoms_2y6w = atoms_2y6w[:min_len]

# Alinear estructuras
super_imposer = Superimposer()
super_imposer.set_atoms(atoms_1ubq, atoms_2y6w)
super_imposer.apply(structure_2y6w.get_atoms())  # Alinear 2Y6W respecto a 1UBQ

# Guardar la estructura alineada
io = PDBIO()
io.set_structure(structure_2y6w)
io.save('2y6w_aligned.pdb')

# Leer el archivo alineado
with open('2y6w_aligned.pdb', 'r') as f:
    pdb_data_2y6w_aligned = f.read()

# Crear visor py3Dmol
viewer = py3Dmol.view(width=600, height=400)

# Cargar proteína 1UBQ
viewer.addModel(pdb_data_1ubq, 'pdb')

# Cargar proteína 2Y6W alineada
viewer.addModel(pdb_data_2y6w_aligned, 'pdb')

# Estilo para cada proteína
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'blue'}})  # 1UBQ en azul
viewer.setStyle({'model': 1}, {'cartoon': {'color': 'red'}})   # 2Y6W en rojo

# Centrar y mostrar
viewer.zoomTo()
viewer.setBackgroundColor('white')
viewer.show()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.2 MB/s eta 0:00:00


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
# Descargar datos PDB
pdb_data_2IQH = get_pdb_data('2IQH')
pdb_data_3RO5 = get_pdb_data('3RO5')

# Guardar datos en archivos temporales para Biopython
with open('2IQH.pdb', 'w') as f:
    f.write(pdb_data_2IQH)
with open('3RO5.pdb', 'w') as f:
    f.write(pdb_data_3RO5)

# Parsear estructuras con Biopython
parser = PDB.PDBParser(QUIET=True)
structure_2IQH = parser.get_structure('2IQH', '2IQH.pdb')
structure_3RO5 = parser.get_structure('3RO5', '3RO5.pdb')

# Obtener átomos de la cadena A (puedes ajustar según la estructura)
atoms_2IQH = [atom for atom in structure_2IQH[0]['A'].get_atoms() if atom.get_name() == 'CA']  # Solo átomos CA
atoms_3RO5 = [atom for atom in structure_3RO5[0]['A'].get_atoms() if atom.get_name() == 'CA']

# Asegurar que ambas listas tengan la misma longitud
min_len = min(len(atoms_2IQH), len(atoms_3RO5))
atoms_2IQH = atoms_2IQH[:min_len]
atoms_3RO5 = atoms_3RO5[:min_len]

# Alinear estructuras
super_imposer = Superimposer()
super_imposer.set_atoms(atoms_2IQH, atoms_3RO5)
super_imposer.apply(structure_3RO5.get_atoms())  # Alinear 3RO5 respecto a 2IQH

# Guardar la estructura alineada
io = PDBIO()
io.set_structure(structure_3RO5)
io.save('3RO5_aligned.pdb')

# Leer el archivo alineado
with open('3RO5_aligned.pdb', 'r') as f:
    pdb_data_3RO5_aligned = f.read()

# Crear visor py3Dmol
viewer = py3Dmol.view(width=600, height=400)

# Cargar proteína 2IQH
viewer.addModel(pdb_data_2IQH, 'pdb')

# Cargar proteína 3RO5 alineada
viewer.addModel(pdb_data_3RO5_aligned, 'pdb')

# Estilo para cada proteína
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'blue'}})  # 2IQH en azul
viewer.setStyle({'model': 1}, {'cartoon': {'color': 'red'}})   # 3RO5 en rojo

# Centrar y mostrar
viewer.zoomTo()
viewer.setBackgroundColor('white')
viewer.show()


Output hidden; open in https://colab.research.google.com to view.

Explicación:
1.  Instalación: Instalamos Biopython con !pip install biopython.
2.  Descarga de PDB: Usamos requests para descargar los archivos PDB de 2IQH y 3RO5.
3.  Parseo con Biopython: Parseamos los archivos PDB con PDB.PDBParser para obtener las estructuras.
4.  Alineación: Usamos Superimposer de Biopython para alinear las estructuras basándonos en los átomos CA (carbonos alfa) de la cadena A. Esto asegura que las proteínas estén superpuestas correctamente.
5.  Guardado: Guardamos la estructura alineada (3RO5) en un nuevo archivo PDB.
6.  Visualización: Cargamos ambas estructuras (2IQH original y 3RO5 alineada) en py3Dmol y las coloreamos diferente para compararlas.
Notas:
•  Ajuste de cadenas: El código asume que ambas proteínas tienen una cadena A. Si 3RO5 tiene una cadena diferente (por ejemplo, B), cambia 'A' por la cadena correcta. Puedes inspeccionar las cadenas con:

In [10]:
for model in structure_3RO5:
    for chain in model:
        print(chain.get_id())

A
B


Longitud de átomos: Recortamos las listas de átomos CA para que tengan la misma longitud, ya que 2IQH y 3RO5 pueden tener diferente número de residuos.
•  Errores posibles: Si las proteínas no son homólogas o tienen estructuras muy diferentes, la alineación puede no ser precisa. En ese caso, busca proteínas más similares o usa solo una región (por ejemplo, un dominio conservado).
Tareas:
•  Prueba con otras proteínas homólogas (por ejemplo, 1GZM y una mutante del PDB).
•  Resalta residuos específicos (por ejemplo, sitios activos) con viewer.addStyle({'resi': 17}, {'stick': {'colorscheme': 'yellowCarbon'}}).
•  Añade una superficie transparente a una de las proteínas con viewer.addSurface(py3Dmol.VDW, {'opacity': 0.7, 'color': 'grey'}, {'model': 0}).
•  Cambia los colores a hexadecimales (por ejemplo, #00FF00 para 3RO5).

Objetivo: Simular un docking molecular sencillo (por ejemplo, salbutamol en el receptor β2-adrenérgico, PDB: 7DHI, como mencionaste antes) y visualizar el resultado en py3Dmol con colores personalizados para resaltar interacciones clave.

In [11]:
#!pip install biopython

import py3Dmol
import requests
from Bio import PDB
from Bio.PDB import PDBIO, Superimposer
import io

# Descargar datos PDB de la proteína (7DHI)
def get_pdb_data(pdb_id):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    response = requests.get(url)
    return response.text

pdb_data_7dhi = get_pdb_data('7DHI')

# Descargar estructura de salbutamol (puedes usar un archivo PDB o XYZ)
# Nota: Como ejemplo, usamos una estructura XYZ simulada para salbutamol
salbutamol_xyz = """
12
Salbutamol (simulated)
C 0.000000 0.000000 0.000000
C 1.400000 0.000000 0.000000
O 1.400000 1.000000 0.000000
N -1.400000 0.000000 0.000000
C -1.400000 -1.400000 0.000000
C 0.000000 -1.400000 0.000000
O 0.000000 -2.400000 0.000000
H 2.400000 1.000000 0.000000
H -2.400000 0.000000 0.000000
H -1.400000 -2.400000 0.000000
H 0.000000 -1.400000 1.000000
H 0.000000 -1.400000 -1.000000
"""

# Crear visor
viewer = py3Dmol.view(width=600, height=400)

# Cargar proteína
viewer.addModel(pdb_data_7dhi, 'pdb')

# Cargar ligando (salbutamol)
viewer.addModel(salbutamol_xyz, 'xyz')

# Estilo para proteína
viewer.setStyle({'model': 0}, {'cartoon': {'color': '#00FFFF'}})  # 7DHI en cian

# Estilo para ligando
viewer.setStyle({'model': 1}, {'stick': {'colorscheme': 'redCarbon'}})

# Resaltar residuos cercanos al ligando (simulamos interacción a < 4 Å)
viewer.addStyle({'model': 0, 'within': {'distance': 4, 'sel': {'model': 1}}},
                {'stick': {'colorscheme': 'yellowCarbon'}})

# Añadir líneas para interacciones
viewer.addStyle({'model': 0, 'within': {'distance': 3.5, 'sel': {'model': 1}}},
                {'line': {'color': 'white'}})

# Centrar y mostrar
viewer.zoomTo()
viewer.setBackgroundColor('black')
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Objetivo: Visualizar un complejo proteico multimérico (por ejemplo, hemoglobina, PDB: 4HHB) y colorear cada subunidad con colores personalizados para analizar su ensamblaje.

In [12]:
#!pip install biopython

import py3Dmol
import requests
from Bio import PDB
import io

# Descargar datos PDB
def get_pdb_data(pdb_id):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    response = requests.get(url)
    return response.text

pdb_data_4hhb = get_pdb_data('4HHB')

# Parsear estructura para identificar cadenas
parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure('4HHB', io.StringIO(pdb_data_4hhb))
chains = [chain.get_id() for chain in structure[0]]
print(f"Cadenas en 4HHB: {chains}")

# Crear visor
viewer = py3Dmol.view(width=600, height=400)

# Cargar proteína
viewer.addModel(pdb_data_4hhb, 'pdb')

# Estilo para cada cadena
colors = {'A': '#FF0000', 'B': '#00FF00', 'C': '#0000FF', 'D': '#FFFF00'}  # Colores por cadena
for chain in chains:
    viewer.setStyle({'chain': chain}, {'cartoon': {'color': colors.get(chain, 'grey')}})

# Resaltar grupos hemo (ligandos)
viewer.addStyle({'hetflag': True}, {'stick': {'colorscheme': 'redCarbon'}})

# Añadir superficie transparente para una cadena
viewer.addSurface(py3Dmol.VDW, {'opacity': 0.7, 'color': 'white'}, {'chain': 'B'})

# Centrar y mostrar
viewer.zoomTo()
viewer.setBackgroundColor('black')
viewer.show()


Cadenas en 4HHB: ['A', 'B', 'C', 'D']


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Ejercicio 16: Alineación y Comparación de Sitios Activos Objetivo: Alinear dos proteínas (1AKI y 1OPK, ambas kinasas) y resaltar sus sitios activos con colores personalizados.

In [13]:
!pip install biopython

import py3Dmol
import requests
from Bio import PDB
from Bio.PDB import Superimposer, PDBIO
import io

# Descargar datos PDB
def get_pdb_data(pdb_id):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    response = requests.get(url)
    return response.text

pdb_data_1aki = get_pdb_data('1AKI')
pdb_data_1opk = get_pdb_data('1OPK')

# Guardar datos en archivos temporales
with open('1aki.pdb', 'w') as f:
    f.write(pdb_data_1aki)
with open('1opk.pdb', 'w') as f:
    f.write(pdb_data_1opk)

# Parsear estructuras
parser = PDB.PDBParser(QUIET=True)
structure_1aki = parser.get_structure('1AKI', '1aki.pdb')
structure_1opk = parser.get_structure('1OPK', '1opk.pdb')

# Obtener átomos CA de la cadena A
atoms_1aki = [atom for atom in structure_1aki[0]['A'].get_atoms() if atom.get_name() == 'CA']
atoms_1opk = [atom for atom in structure_1opk[0]['A'].get_atoms() if atom.get_name() == 'CA']

# Ajustar longitud
min_len = min(len(atoms_1aki), len(atoms_1opk))
atoms_1aki = atoms_1aki[:min_len]
atoms_1opk = atoms_1opk[:min_len]

# Alinear estructuras
super_imposer = Superimposer()
super_imposer.set_atoms(atoms_1aki, atoms_1opk)
super_imposer.apply(structure_1opk.get_atoms())

# Guardar estructura alineada
io = PDBIO()
io.set_structure(structure_1opk)
io.save('1opk_aligned.pdb')

# Leer archivo alineado
with open('1opk_aligned.pdb', 'r') as f:
    pdb_data_1opk_aligned = f.read()

# Crear visor
viewer = py3Dmol.view(width=600, height=400)

# Cargar proteínas
viewer.addModel(pdb_data_1aki, 'pdb')
viewer.addModel(pdb_data_1opk_aligned, 'pdb')

# Estilo para proteínas
viewer.setStyle({'model': 0}, {'cartoon': {'color': '#00FFFF'}})  # 1AKI en cian
viewer.setStyle({'model': 1}, {'cartoon': {'color': '#FF00FF'}})  # 1OPK en magenta

# Resaltar sitios activos (residuos aproximados, ajusta según literatura)
viewer.addStyle({'model': 0, 'resi': '139-142'}, {'stick': {'colorscheme': 'yellowCarbon'}})
viewer.addStyle({'model': 1, 'resi': '139-142'}, {'stick': {'colorscheme': 'greenCarbon'}})

# Mostrar
viewer.zoomTo()
viewer.setBackgroundColor('black')
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Visualización de un Ensamblaje con Residuos Clave Resaltados

Cargar la hemoglobina (4HHB), alinear dos copias con una mutación simulada en cada subunidad, y visualizarlas con colores por cadena y animación de rotación.

In [23]:
#!pip install biopython

import py3Dmol
import requests
from Bio import PDB
from Bio.PDB import Superimposer, PDBIO
import io

# Descargar datos PDB
def get_pdb_data(pdb_id):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url)
    return response.text

pdb_data_4hhb = get_pdb_data("4HHB")

# Parsear estructura
parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure("4HHB", io.StringIO(pdb_data_4hhb))
structure_mutant = parser.get_structure("4HHB_mutant", io.StringIO(pdb_data_4hhb))

# Mutar residuo 6 en cadena A (por ejemplo, de GLU a VAL, como en anemia falciforme)
for model in structure_mutant:
    for chain in model:
        if chain.get_id() == "A" and 6 in chain:
            residue = chain[6]
            if residue.get_resname() == "GLU":
                residue.resname = "VAL"
                atoms_to_remove = [atom.get_id() for atom in residue if atom.get_id() not in ["N", "CA", "C", "O", "CB"]]
                for atom_id in atoms_to_remove:
                    residue.detach_child(atom_id)

# Alinear estructuras
atoms_orig = [atom for atom in structure[0]["A"].get_atoms() if atom.get_name() == "CA"]
atoms_mut = [atom for atom in structure_mutant[0]["A"].get_atoms() if atom.get_name() == "CA"]
min_len = min(len(atoms_orig), len(atoms_mut))
atoms_orig = atoms_orig[:min_len]
atoms_mut = atoms_mut[:min_len]
super_imposer = Superimposer()
super_imposer.set_atoms(atoms_orig, atoms_mut)
super_imposer.apply(structure_mutant.get_atoms())

# Guardar estructura mutante
io_writer = PDBIO()
io_writer.set_structure(structure_mutant)
mutant_pdb_io = io.StringIO()
io_writer.save(mutant_pdb_io)
pdb_data_4hhb_mutant = mutant_pdb_io.getvalue()

# Crear visor
viewer = py3Dmol.view(width=600, height=400)

# Cargar estructuras
viewer.addModel(pdb_data_4hhb, "pdb")
viewer.addModel(pdb_data_4hhb_mutant, "pdb")

# Estilo por cadenas
colors = {"A": "#FF0000", "B": "#00FF00", "C": "#0000FF", "D": "#FFFF00"}
for chain in colors:
    viewer.setStyle({"model": 0, "chain": chain}, {"cartoon": {"color": colors[chain]}})
    viewer.setStyle({"model": 1, "chain": chain}, {"cartoon": {"color": colors[chain]}})

# Resaltar residuo 6 en cadena A
viewer.addStyle({"model": 0, "chain": "A", "resi": 6}, {"stick": {"colorscheme": "yellowCarbon"}})
viewer.addStyle({"model": 1, "chain": "A", "resi": 6}, {"stick": {"colorscheme": "greenCarbon"}})

# Estilo para grupos hemo
viewer.addStyle({"hetflag": True}, {"stick": {"colorscheme": "redCarbon"}})

# Añadir animación de rotación
# viewer.animate({"loop": "forward", "interval": 50})

# Verificar modelos
print(f"Número de modelos cargados: {viewer.getNumberOfModels()}")

# Mostrar
viewer.zoomTo()
viewer.setBackgroundColor("black")
viewer.show()


Número de modelos cargados: <py3Dmol.view object at 0x7c7b4e0110d0>


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Explicación:
•  Proteína: Usamos 4HHB (hemoglobina), mutamos el residuo 6 de la cadena A (GLU a VAL, relevante para anemia falciforme), y alineamos las estructuras.
•  Visualización: Coloreamos cada cadena (A, B, C, D) con colores diferentes, resaltamos el residuo 6, y mostramos los grupos hemo.
•  Animación: Añadimos una rotación suave para ver el ensamblaje desde todos los ángulos.
•  Depuración: Verificamos la carga de modelos.
Tareas:
•  Muta otro residuo en otra cadena (por ejemplo, residuo 6 en cadena B).
•  Cambia la animación a un bucle de modelos (mods: [0, 1]) en lugar de rotación.
•  Añade una superficie a una cadena con viewer.addSurface(py3Dmol.VDW, {"opacity": 0.7, "color": "grey"}, {"chain": "A"}).
•  Prueba con otro complejo multimérico (por ejemplo, 1AON).

In [25]:
import py3Dmol
import requests
from Bio import PDB
from Bio.PDB import Superimposer, PDBIO
import io

# Descargar datos PDB
def get_pdb_data(pdb_id):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Could not download {pdb_id}")
    return response.text

# PDB ID
pdb_id = "4HHB"
pdb_data_4hhb = get_pdb_data(pdb_id)

# Parsear estructura original
parser = PDB.PDBParser(QUIET=True)
structure_original = parser.get_structure(pdb_id, io.StringIO(pdb_data_4hhb))

# Crear una copia de la estructura para mutación
structure_mutated = parser.get_structure(f"{pdb_id}_mutant", io.StringIO(pdb_data_4hhb))

# Mutar residuo 6 en cadena B (por ejemplo, de GLU a VAL)
# Asumiendo que 4HHB tiene un GLU en la posición 6 de la cadena B
for model in structure_mutated:
    for chain in model:
        if chain.get_id() == "B" and 6 in chain:
            residue = chain[6]
            # Check if the residue is GLU before attempting to change
            if residue.get_resname() == "GLU":
                residue.resname = "VAL"
                # Remove side chain atoms specific to GLU to make it VAL-like
                # This is a simplified mutation; a real mutation would re-build the side chain
                atoms_to_remove = [atom.get_id() for atom in residue if atom.get_id() not in ["N", "CA", "C", "O", "CB"]]
                for atom_id in atoms_to_remove:
                    residue.detach_child(atom_id)
                print(f"Mutated residue {residue.get_resname()} at chain B, position 6 to VAL")
            else:
                print(f"Residue at chain B, position 6 is not GLU. It is {residue.get_resname()}. No mutation performed.")

# Alinear estructuras (usando la cadena A como referencia para ambas)
# atoms_orig = [atom for atom in structure_original[0]["A"].get_atoms() if atom.get_name() == "CA"]
# atoms_mut = [atom for atom in structure_mutated[0]["A"].get_atoms() if atom.get_name() == "CA"]

# For alignment, we should align the full mutated structure against the original if we want to visualize difference
# Or align the mutated structure back onto itself based on original, to ensure they are in the same frame
# Let's re-parse to ensure the original for alignment is fresh
structure_align_ref = parser.get_structure(pdb_id, io.StringIO(pdb_data_4hhb))

atoms_ref = [atom for atom in structure_align_ref.get_atoms() if atom.get_name() == "CA"]
atoms_mutated_for_align = [atom for atom in structure_mutated.get_atoms() if atom.get_name() == "CA"]

# Ensure both lists have the same length for alignment
min_len_align = min(len(atoms_ref), len(atoms_mutated_for_align))
atoms_ref = atoms_ref[:min_len_align]
atoms_mutated_for_align = atoms_mutated_for_align[:min_len_align]

super_imposer = Superimposer()
super_imposer.set_atoms(atoms_ref, atoms_mutated_for_align)
super_imposer.apply(structure_mutated.get_atoms())

# Guardar estructura mutada alineada en un string
io_writer = PDBIO()
io_writer.set_structure(structure_mutated)
mutated_pdb_io = io.StringIO()
io_writer.save(mutated_pdb_io)
pdb_data_4hhb_mutated = mutated_pdb_io.getvalue()

# Crear visor py3Dmol
viewer = py3Dmol.view(width=600, height=400)

# Cargar estructuras
viewer.addModel(pdb_data_4hhb, "pdb") # Model 0: Original
viewer.addModel(pdb_data_4hhb_mutated, "pdb") # Model 1: Mutated

# Estilo por cadenas para ambas modelos
colors = {"A": "#FF0000", "B": "#00FF00", "C": "#0000FF", "D": "#FFFF00"} # Rojo, Verde, Azul, Amarillo
for chain_id, color in colors.items():
    viewer.setStyle({"model": 0, "chain": chain_id}, {"cartoon": {"color": color}})
    viewer.setStyle({"model": 1, "chain": chain_id}, {"cartoon": {"color": color}})

# Resaltar residuo 6 en cadena A (original) y cadena B (mutado)
# Residuo 6 en cadena A del modelo original
viewer.addStyle({"model": 0, "chain": "A", "resi": 6}, {"stick": {"colorscheme": "yellowCarbon"}})
# Residuo 6 en cadena B del modelo mutado
viewer.addStyle({"model": 1, "chain": "B", "resi": 6}, {"stick": {"colorscheme": "greenCarbon"}})

# Estilo para grupos hemo en ambos modelos
viewer.addStyle({"hetflag": True}, {"stick": {"colorscheme": "redCarbon"}})

# Añadir superficie a una cadena del modelo original (cadena A)
viewer.addSurface(py3Dmol.VDW, {"opacity": 0.7, "color": "grey"}, {"model": 0, "chain": "A"})

# Cambiar animación a bucle de modelos (original y mutado)
viewer.animate({"loop": "forward", "interval": 2000, "steps": [0, 1]})

# Mostrar
viewer.zoomTo()
viewer.setBackgroundColor("black")
viewer.show()

Mutated residue VAL at chain B, position 6 to VAL


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [16]:
import py3Dmol
import requests
from Bio import PDB
from Bio.PDB import Superimposer, PDBIO
import io

# Descargar datos PDB
def get_pdb_data(pdb_id):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Could not download {pdb_id}")
    return response.text

# PDB ID (cambiado a 1AON)
pdb_id = "1AON"
pdb_data_1aon = get_pdb_data(pdb_id)

# Parsear estructura original
parser = PDB.PDBParser(QUIET=True)
structure_original = parser.get_structure(pdb_id, io.StringIO(pdb_data_1aon))

# Crear una copia de la estructura para mutación
structure_mutated = parser.get_structure(f"{pdb_id}_mutant", io.StringIO(pdb_data_1aon))

# Mutar residuo 6 en cadena B (por ejemplo, de GLU a VAL)
# Asumiendo que el PDB 1AON tiene un GLU en la posición 6 de la cadena B
for model in structure_mutated:
    for chain in model:
        if chain.get_id() == "B" and 6 in chain:
            residue = chain[6]
            # Check if the residue is GLU before attempting to change
            if residue.get_resname() == "GLU":
                residue.resname = "VAL"
                # Remove side chain atoms specific to GLU to make it VAL-like
                # This is a simplified mutation; a real mutation would re-build the side chain
                atoms_to_remove = [atom.get_id() for atom in residue if atom.get_id() not in ["N", "CA", "C", "O", "CB"]]
                for atom_id in atoms_to_remove:
                    residue.detach_child(atom_id)
                print(f"Mutated residue {residue.get_resname()} at chain B, position 6 to VAL")
            else:
                print(f"Residue at chain B, position 6 is not GLU. It is {residue.get_resname()}. No mutation performed.")

# Alinear estructuras
structure_align_ref = parser.get_structure(pdb_id, io.StringIO(pdb_data_1aon))

atoms_ref = [atom for atom in structure_align_ref.get_atoms() if atom.get_name() == "CA"]
atoms_mutated_for_align = [atom for atom in structure_mutated.get_atoms() if atom.get_name() == "CA"]

# Ensure both lists have the same length for alignment
min_len_align = min(len(atoms_ref), len(atoms_mutated_for_align))
atoms_ref = atoms_ref[:min_len_align]
atoms_mutated_for_align = atoms_mutated_for_align[:min_len_align]

super_imposer = Superimposer()
super_imposer.set_atoms(atoms_ref, atoms_mutated_for_align)
super_imposer.apply(structure_mutated.get_atoms())

# Guardar estructura mutada alineada en un string
io_writer = PDBIO()
io_writer.set_structure(structure_mutated)
mutated_pdb_io = io.StringIO()
io_writer.save(mutated_pdb_io)
pdb_data_1aon_mutated = mutated_pdb_io.getvalue()

# Crear visor py3Dmol
viewer = py3Dmol.view(width=600, height=400)

# Cargar estructuras
viewer.addModel(pdb_data_1aon, "pdb") # Model 0: Original
viewer.addModel(pdb_data_1aon_mutated, "pdb") # Model 1: Mutated

# Obtener cadenas presentes en 1AON para colorear dinámicamente
chains_1aon = [chain.get_id() for chain in structure_original[0]]

# Estilo por cadenas para ambas modelos
# Usar un conjunto de colores predefinidos o generar dinámicamente
def get_chain_colors(num_chains):
    predefined_colors = ["#FF0000", "#00FF00", "#0000FF", "#FFFF00", "#FF00FF", "#00FFFF", "#FFA500", "#800080"]
    return {chain_id: predefined_colors[i % len(predefined_colors)] for i, chain_id in enumerate(sorted(chains_1aon))}

colors = get_chain_colors(len(chains_1aon))

for chain_id in chains_1aon:
    color = colors.get(chain_id, 'grey') # Default to grey if not in predefined colors
    viewer.setStyle({"model": 0, "chain": chain_id}, {"cartoon": {"color": color}})
    viewer.setStyle({"model": 1, "chain": chain_id}, {"cartoon": {"color": color}})

# Resaltar residuo 6 en cadena B (mutado)
viewer.addStyle({"model": 1, "chain": "B", "resi": 6}, {"stick": {"colorscheme": "greenCarbon"}})

# Estilo para grupos hemo/ligandos (si los hay en 1AON)
viewer.addStyle({"hetflag": True}, {"stick": {"colorscheme": "redCarbon"}})

# Añadir superficie a una cadena del modelo original (cadena A)
viewer.addSurface(py3Dmol.VDW, {"opacity": 0.7, "color": "grey"}, {"model": 0, "chain": "A"})

# Cambiar animación a bucle de modelos (original y mutado)
viewer.animate({"loop": "forward", "interval": 2000, "steps": [0, 1]})

# Mostrar
viewer.zoomTo()
viewer.setBackgroundColor("black")
viewer.show()

Output hidden; open in https://colab.research.google.com to view.

In [30]:
import requests
from Bio import PDB
import io

def get_pdb_data(pdb_id):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    response = requests.get(url)
    return response.text

pdb_id_code = '2ZHT'
pdb_content = get_pdb_data(pdb_id_code)

parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure(pdb_id_code, io.StringIO(pdb_content))

cysteine_residues = []
for model in structure:
    for chain in model:
        for residue in chain:
            if residue.get_resname() == 'CYS':
                cysteine_residues.append(residue.get_id()[1]) # get_id() returns a tuple, [1] is the residue number

print(f"Cisteína residuos en la proteína {pdb_id_code}: {sorted(list(set(cysteine_residues)))}")

Cisteína residuos en la proteína 2ZHT: [155, 217, 269, 319, 359, 382]
